# EDA de huellas de edificios — Jose Eduardo

Análisis exploratorio de `edificios_google` y `edificios_microsoft` (semana 3).
Cubre **e14** (armado con muestra), **e15** (estadísticas por fuente),
**e16** (comparación entre fuentes) y **e17** (mapa de control de edificios).

**Entradas:** `data/processed/edificios_google.parquet` y `data/processed/edificios_microsoft.parquet`,
que generan Jorge y María Camila (limpieza) con el contrato del Entrega 1 (sección 5.2).
Los límites PDET salen de `data/processed/municipios_pdet.geojson`, que ya está en el repo.

**Variables de entorno (opcionales):**
- `EDA_DATOS`: carpeta con los Parquet (por defecto `data/processed`).
- `EDA_MUESTRA_N`: número de edificios por fuente para una muestra rápida (0 = todos).

In [ ]:
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATOS = Path(os.environ.get("EDA_DATOS", RAIZ / "data" / "processed"))
MUESTRA_N = int(os.environ.get("EDA_MUESTRA_N", "0"))
RUTAS = {
    "google": DATOS / "edificios_google.parquet",
    "microsoft": DATOS / "edificios_microsoft.parquet",
}
faltantes = [str(r) for r in RUTAS.values() if not r.exists()]
if faltantes:
    raise FileNotFoundError(
        "Faltan Parquet para el EDA (los generan Jorge y María Camila):\n  " + "\n  ".join(faltantes)
    )
print("Modo:", f"muestra de {MUESTRA_N:,} por fuente" if MUESTRA_N else "datos completos")

## Municipios PDET

Se usan para asignar cada edificio a su subregión (por centroide) y para el mapa.
El GeoJSON se lee a mano porque `subregion_pdet` es un objeto anidado `{id, nombre}`.

In [ ]:
with open(RAIZ / "data" / "processed" / "municipios_pdet.geojson", encoding="utf-8") as f:
    feats = json.load(f)["features"]
municipios = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")
municipios["subregion"] = [f["properties"]["subregion_pdet"]["nombre"] for f in feats]
municipios["_id"] = [f["properties"]["_id"] for f in feats]
print(len(municipios), "municipios PDET,", municipios["subregion"].nunique(), "subregiones")

## e14 · Carga con muestra

Se revisa el contrato antes de calcular nada: si una columna falta, el notebook se detiene aquí.

In [ ]:
COLUMNAS = ["area_m2", "geom_hash"]
COLUMNAS_GOOGLE = ["confianza", "area_m2_fuente"]

def cargar(ruta):
    gdf = gpd.read_parquet(ruta)
    if MUESTRA_N and len(gdf) > MUESTRA_N:
        gdf = gdf.sample(MUESTRA_N, random_state=42)
    if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
    return gdf

gdfs = {}
for fuente, ruta in RUTAS.items():
    gdf = cargar(ruta)
    requeridas = COLUMNAS + (COLUMNAS_GOOGLE if fuente == "google" else [])
    falta = [c for c in requeridas if c not in gdf.columns]
    if falta:
        raise KeyError(f"{fuente}: faltan columnas del contrato {falta}. Columnas: {list(gdf.columns)}")
    gdfs[fuente] = gdf
    print(f"{fuente:10s} {len(gdf):>12,} registros  columnas: {list(gdf.columns)}")

In [ ]:
def puntos(gdf):
    # Centroide almacenado (geometría o WKB) si existe; si no, centroide de la huella.
    if "centroide" in gdf.columns:
        c = gdf["centroide"]
        if c.dtype == object and len(c) and isinstance(c.iloc[0], (bytes, bytearray)):
            c = gpd.GeoSeries(shapely.from_wkb(c.values), index=gdf.index)
        return gpd.GeoSeries(c, index=gdf.index, crs=4326)
    return gdf.geometry.centroid

def con_subregion(gdf):
    pts = gpd.GeoDataFrame(geometry=puntos(gdf), crs=4326)
    return gpd.sjoin(pts, municipios[["subregion", "_id", "geometry"]], how="inner", predicate="within")

# Duplicados entre particiones: no deberían existir tras la limpieza.
for k, gdf in gdfs.items():
    print(k, "geom_hash duplicados:", int(gdf["geom_hash"].duplicated().sum()))

## e15 · Estadísticas por fuente

In [ ]:
def resumen(gdf, nombre):
    a = gdf["area_m2"]
    return pd.Series({
        "fuente": nombre,
        "edificios": len(gdf),
        "area_media_m2": a.mean(),
        "area_mediana_m2": a.median(),
        "p10_m2": a.quantile(0.10),
        "p90_m2": a.quantile(0.90),
        "p99_m2": a.quantile(0.99),
        "area_max_m2": a.max(),
        "area_nula": int(a.isna().sum()),
    })

tabla_estadisticas = pd.DataFrame([resumen(g, k) for k, g in gdfs.items()]).set_index("fuente")
tabla_estadisticas

In [ ]:
g = gdfs["google"]
print("Confianza (Google):")
print(g["confianza"].describe().to_string())
print("\nBajo 0,65 (fuera del rango del Entrega 1):", int((g["confianza"] < 0.65).sum()))

In [ ]:
conteos = pd.concat({k: con_subregion(g).groupby("subregion").size() for k, g in gdfs.items()}, axis=1).fillna(0).astype(int)
conteos.columns = ["Google", "Microsoft"]
print("Edificios por subregión PDET:")
conteos.sort_values("Google", ascending=False)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))

# Distribución de áreas (escala log).
for k, g in gdfs.items():
    ejes[0].hist(np.log10(g["area_m2"].clip(lower=1)), bins=80, histtype="step", linewidth=1.5, label=k, density=True)
ejes[0].set_xlabel("log10(área m²)")
ejes[0].set_title("Distribución de áreas de huella")
ejes[0].legend()

# Confianza de Google.
ejes[1].hist(gdfs["google"]["confianza"].dropna(), bins=70, color="#4c72b0")
ejes[1].set_xlabel("confianza (Google)")
ejes[1].set_title("Distribución de confianza, Google")

plt.tight_layout()
plt.show()

fig, eje = plt.subplots(figsize=(12, 4.5))
conteos.sort_values("Google", ascending=False).plot.bar(ax=eje)
eje.set_ylabel("edificios")
eje.set_title("Edificios por subregión PDET")
plt.tight_layout()
plt.show()

## e16 · Comparación entre fuentes

Conteo y área total por fuente, y para Google la diferencia entre `area_m2_fuente` (la que reporta Google) y `area_m2` (la nuestra, en EPSG:9377).

In [ ]:
totales = pd.DataFrame({
    "edificios": {k: len(g) for k, g in gdfs.items()},
    "area_total_km2": {k: g["area_m2"].sum() / 1e6 for k, g in gdfs.items()},
})
totales["area_total_km2"] = totales["area_total_km2"].round(3)
print("Totales (sobre los datos cargados:", "muestra" if MUESTRA_N else "completos", ")")
totales

In [ ]:
g = gdfs["google"].dropna(subset=["area_m2_fuente"]).copy()
g["dif_rel"] = (g["area_m2"] - g["area_m2_fuente"]) / g["area_m2_fuente"]
print("Registros comparables (Google):", f"{len(g):,}")
print("Correlación area_m2 vs area_m2_fuente:", round(g[["area_m2", "area_m2_fuente"]].corr().iloc[0, 1], 4))
print("\nDiferencia relativa (nuestra - fuente) / fuente:")
print(g["dif_rel"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_string())

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))

muestra_graf = g.sample(min(50_000, len(g)), random_state=42)
ejes[0].scatter(muestra_graf["area_m2_fuente"], muestra_graf["area_m2"], s=2, alpha=0.3)
lim = [1, max(muestra_graf[["area_m2", "area_m2_fuente"]].max())]
ejes[0].plot(lim, lim, color="red", linewidth=1)
ejes[0].set_xscale("log"); ejes[0].set_yscale("log")
ejes[0].set_xlabel("area_m2_fuente (Google)"); ejes[0].set_ylabel("area_m2 (nuestra)")
ejes[0].set_title("Área: Google vs nuestra medición")

ejes[1].hist(g["dif_rel"].clip(-0.5, 0.5), bins=100, color="#dd8452")
ejes[1].set_title("Diferencia relativa (recortada a ±50 %)")

ejes[2].bar(list(totales.index), totales["edificios"], color=["#4c72b0", "#55a868"])
ejes[2].set_title("Edificios por fuente")

plt.tight_layout()
plt.show()

## e17 · Mapa de control de edificios

Densidad de centroides por fuente sobre los municipios PDET. Sirve para ver huecos de cobertura:
municipios sin edificios o con muy pocos.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True)
for eje, (k, gdf) in zip(ejes, gdfs.items()):
    municipios.boundary.plot(ax=eje, linewidth=0.3, color="0.6")
    p = puntos(gdf)
    hb = eje.hexbin(p.x, p.y, gridsize=140, bins="log", mincnt=1, cmap="viridis")
    eje.set_title(f"Densidad de edificios: {k}")
    eje.set_aspect("equal")
fig.colorbar(hb, ax=ejes, shrink=0.7, label="edificios por celda (log)")
plt.show()

In [ ]:
cobertura = pd.DataFrame(index=municipios["_id"])
for k, gdf in gdfs.items():
    pts = gpd.GeoDataFrame(geometry=puntos(gdf), crs=4326)
    j = gpd.sjoin(pts, municipios[["_id", "geometry"]], how="inner", predicate="within")
    cobertura[k] = j.groupby("_id").size().reindex(cobertura.index, fill_value=0).values

sin = cobertura[(cobertura == 0).any(axis=1)]
print(f"Municipios sin edificios en alguna fuente: {len(sin)}")
cobertura.assign(nombre=municipios.set_index("_id")["nombre"]).sort_values("google").head(15)

---
**Notas para el informe:** si `EDA_MUESTRA_N` está activo, todas las cifras son de la muestra y no deben citarse como resultados.